# **CH07 텍스트 분할(Text Splitter)**
<hr/>

#### __문서분할 과정__

1. 문서 구조 파악: PDF 파일, 웹 페이지, 전자 책 등 다양한 형식의 문서에서 구조를 파악합니다. 이는 문서의 헤더, 푸터, 페이지 번호, 섹션 제목 등을 식별하는 과정을 포함할 수 있습니다.
2. 단위 선정: 문서를 어떤 단위로 나눌지 결정합니다. 이는 페이지별, 섹션별, 또는 문단별일 수 있으며, 문서의 내용과 목적에 따라 다릅니다.
3. 단위 크기 선정(chunk size): 문서를 몇 개의 토큰 단위로 나눌 것인지를 정합니다.
4. 청크 오버랩(chunk overlap): 분할된 끝 부분에서 맥락이 이어질 수 있도록 일부를 겹쳐서(overlap) 분할하는 것이 일반적입니다.

* [Chunk Visualization 사이트](https://chunkviz.up.railway.app/)

### __1. 문자 텍스트 분할(CharacterTextSplitter)__

* 기본적으로 "\n\n" 을 기준으로 문자 단위로 텍스트를 분할하고, 청크의 크기를 문자 수로 측정합니다.

  > **텍스트 분할 방식**: 단일 문자 기준   
  > **청크 크기 측정 방식**: 문자 수 기준

In [ ]:
# Text Splitter Library 설치
!pip install -qU langchain-text-splitters

In [ ]:
# ./data/appendix-keywords.txt 파일을 열어서 f라는 파일 객체를 생성합니다.
with open("./data/appendix-keywords.txt", encoding="UTF-8") as f:
    file = f.read()  # 파일의 내용을 읽어서 file 변수에 저장합니다.

    # 파일으로부터 읽은 내용을 일부 출력합니다.
    print(file[:500])

**CharacterTextSplitter를 사용하여 텍스트를 청크(chunk)로 분할**

| Param | Desc |
| :--- | :--- |
| __separator__ | 매개변수로 분할할 기준을 설정합니다. 기본 값은 "\n\n" 입니다.
| __chunk_size__ | 매개변수를 250 으로 설정하여 각 청크의 최대 크기를 250자로 제한합니다.
| __chunk_overlap__ | 매개변수를 50으로 설정하여 인접한 청크 간에 50자의 중복을 허용합니다.
| __length_function__ | 매개변수를 len으로 설정하여 텍스트의 길이를 계산하는 함수를 지정합니다.
| __is_separator_regex__ | 매개변수를 False로 설정하여 separator를 정규식이 아닌 일반 문자열로 처리합니다.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

# TextSplitter 설정
text_splitter = CharacterTextSplitter(
    # 텍스트를 분할할 때 사용할 구분자를 지정합니다. 기본값은 "\n\n"입니다.
    # separator=" ",
    # 분할된 텍스트 청크의 최대 크기를 지정합니다.
    chunk_size=250,
    # 분할된 텍스트 청크 간의 중복되는 문자 수를 지정합니다.
    chunk_overlap=50,
    # 텍스트의 길이를 계산하는 함수를 지정합니다.
    length_function=len,
    # 구분자가 정규식인지 여부를 지정합니다.
    is_separator_regex=False,
)

# text_splitter를 사용하여 state_of_the_union 텍스트를 문서로 분할합니다.
texts = text_splitter.create_documents([file])
print(texts[0])  # 분할된 문서 중 첫 번째 문서를 출력합니다.

print("\n*** 분할된 문서 수: ", len(texts))

#texts

In [ ]:
###################### 메타데이터 지정

metadatas = [
    {"document": 1},
    {"document": 2},
]  # 문서에 대한 메타데이터 리스트를 정의합니다.
documents = text_splitter.create_documents(
    [
        file,
        file,
    ],  # 분할할 텍스트 데이터를 리스트로 전달합니다.
    metadatas=metadatas,  # 각 문서에 해당하는 메타데이터를 전달합니다.
)
print(documents[0])  # 분할된 문서 중 첫 번째 문서를 출력합니다.

In [ ]:
# text_splitter를 사용하여 file 텍스트를 분할하고, 분할된 텍스트의 첫 번째 요소를 반환합니다.
doc = text_splitter.split_text(file)[0]

print(doc)
print ("---------------------------")
print("")

for idx, text in enumerate(text_splitter.split_text(file)):
    print("[", idx, "]", "==============================")
    print(text, "\n")

### __2. 재귀적 문자 텍스트 분할(RecursiveCharacterTextSplitter)__

In [ ]:
!pip install -qU langchain-text-splitters

In [ ]:
# appendix-keywords.txt 파일을 열어서 f라는 파일 객체를 생성합니다.
with open("./data/appendix-keywords.txt", encoding="UTF-8") as f:
    file = f.read()  # 파일의 내용을 읽어서 file 변수에 저장합니다.

# 파일으로부터 읽은 내용을 일부 출력합니다.
print(file[:500])    

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Text Splitter 설정
text_splitter = RecursiveCharacterTextSplitter(
    # 청크 크기를 매우 작게 설정합니다. 예시를 위한 설정입니다.
    chunk_size=250,
    # 청크 간의 중복되는 문자 수를 설정합니다.
    chunk_overlap=50,
    # 문자열 길이를 계산하는 함수를 지정합니다.
    length_function=len,
    # 구분자로 정규식을 사용할지 여부를 설정합니다.
    is_separator_regex=False,
)

# text_splitter를 사용하여 file 텍스트를 문서로 분할합니다.
texts = text_splitter.create_documents([file])
print(texts[0])  # 분할된 문서의 첫 번째 문서를 출력합니다.
print("===" * 20)
print(texts[1])  # 분할된 문서의 두 번째 문서를 출력합니다.

In [ ]:
# 텍스트를 분할하고 분할된 텍스트의 처음 2개 요소를 반환합니다.
text_splitter.split_text(file)[:2]

### __3. 토큰 텍스트 분할(TokenTextSplitter)__

* 텍스트를 토큰 수를 기반으로 청크를 생성할 때 유용
* OpenAI API 사용

#### **(1) tiktoken**

* OpenAI에서 만든 빠른 BPE Tokenizer -> 안됨 ㅜㅜ

In [ ]:
!pip install -qU tiktoken

In [ ]:
pip install -U urllib3

In [ ]:
# data/appendix-keywords.txt 파일을 열어서 f라는 파일 객체를 생성합니다.
with open("./data/appendix-keywords.txt", encoding="UTF-8") as f:
    file = f.read()  # 파일의 내용을 읽어서 file 변수에 저장합니다.

# 파일으로부터 읽은 내용을 일부 출력합니다.
print(file[:500])

* __`from_tiktoken_encoder`__ 메서드를 사용하여 Tiktoken 인코더 기반의 텍스트 분할기를 초기화

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

# Text Splitter 설정
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    # 청크 크기를 300으로 설정합니다.
    chunk_size=300,
    # 청크 간 중복되는 부분이 없도록 설정합니다.
    chunk_overlap=0,
)
# file 텍스트를 청크 단위로 분할합니다.
texts = text_splitter.split_text(file)

print(len(texts))  # 분할된 청크의 개수를 출력합니다.

# texts 리스트의 첫 번째 요소를 출력합니다.
print(texts[0])

In [ ]:
from langchain_text_splitters import TokenTextSplitter

text_splitter = TokenTextSplitter(
    chunk_size=200,  # 청크 크기를 10으로 설정합니다.
    chunk_overlap=0,  # 청크 간 중복을 0으로 설정합니다.
)

# state_of_the_union 텍스트를 청크로 분할합니다.
texts = text_splitter.split_text(file)
print(texts[0])  # 분할된 텍스트의 첫 번째 청크를 출력합니다.

#### __(2) SentenceTransformers__

* huggingface.co 연동 (로컬에서 호출 X)
* default model: sentence-transformers/all-mpnet-base-v2

In [ ]:
#########################################
# Model Download & Use Local Model
#########################################
from sentence_transformers import SentenceTransformer

# local 설치 위치 -> C:\Users\사용자\.cache\huggingface\hub\모델명
model_path = "./local_st_model"
# Download and save the model once with an internet connection
#model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
#model.save(model_path)

# Use the local path in the splitter
text_splitter_local = SentenceTransformersTokenTextSplitter(
    model_name=model_path,
    tokens_per_chunk=512,
    chunk_overlap=50
)

In [ ]:
from langchain_text_splitters import SentenceTransformersTokenTextSplitter

# Model 지정
#model_name="sentence-transformers/all-mpnet-base-v2"   #default model
# Local Model Path 지정 -> config.json 파일 위치
model_path="./.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf"

# 문장 분할기를 생성하고 청크 간 중복을 0으로 설정합니다.
splitter = SentenceTransformersTokenTextSplitter(
    model_name=model_path,    
    chunk_size=200,
    chunk_overlap=0,    
)

# data/appendix-keywords.txt 파일을 열어서 f라는 파일 객체를 생성합니다.
with open("./data/appendix-keywords.txt", encoding="UTF-8") as f:
    file = f.read()  # 파일의 내용을 읽어서 file 변수에 저장합니다.

# 파일으로부터 읽은 내용을 일부 출력합니다.
print(file[:350])

###############################
# file 변수에 담긴 텍스트의 토큰의 개수를 세는 코드입니다. 시작과 종료 토큰의 개수를 제외한 후 출력
###############################
count_start_and_stop_tokens = 2  # 시작과 종료 토큰의 개수를 2로 설정합니다.

# 텍스트의 토큰 개수에서 시작과 종료 토큰의 개수를 뺍니다.
text_token_count = splitter.count_tokens(
    text=file) - count_start_and_stop_tokens
print("\n" + "계산된 텍스트 토큰 개수 -->")
print(text_token_count)  # 계산된 텍스트 토큰 개수를 출력합니다.

###############################
#텍스트를 청크(chunk) 단위로 분할  -> 한글의 자음, 모음으로 분리되는 경우가 발생함!
###############################
text_chunks = splitter.split_text(text=file)  # 텍스트를 청크로 분할합니다.

# 0번째 청크를 출력합니다.
print("\n" + "0번째 청크의 출력 -------------->")
print(text_chunks[0])
print("\n" + "1번째 청크의 출력 -------------->")
print(text_chunks[1])
print("\n" + "2번째 청크의 출력 -------------->")
print(text_chunks[2])

#### __(3) NLTK__

* Natural Language Toolkit (NLTK)
* 영어 자연어 처리(NLP)를 위한 라이브러리
* 단순히 "\n\n"으로 분할하는 대신, NLTK tokenizers를 기반으로 텍스트를 분할
  > 텍스트 분할 방법: NLTK tokenizer에 의해 분할<br>
  > chunk 크기 측정 방법: 문자 수에 의해 측정

In [ ]:
# NLTK 설치
!pip install -qU nltk

In [ ]:
# data/appendix-keywords.txt 파일을 열어서 f라는 파일 객체를 생성.
with open("./data/appendix-keywords.txt", encoding="UTF-8") as f:
    file = f.read()  # 파일의 내용을 읽어서 file 변수에 저장.

# 파일으로부터 읽은 내용을 일부 출력합니다.
print(file[:350])

In [ ]:
from langchain_text_splitters import NLTKTextSplitter

# Text Splitter 생성
text_splitter = NLTKTextSplitter(
    chunk_size=200,  # 청크 크기를 200으로 설정합니다.
    chunk_overlap=0,  # 청크 간 중복을 0으로 설정합니다.
)

# text_splitter를 사용하여 file 텍스트를 분할.
texts = text_splitter.split_text(file)
print(texts[0])  # 분할된 텍스트의 첫 번째 요소를 출력.

#### (4) KoNLPy

* KoNLPy(Korean NLP in Python)는 한국어 자연어 처리(NLP)를 위한 파이썬 패키지
* 문장을 단어로, 단어를 각각의 형태소로 분해하고 각 토큰에 대한 품사를 식별
* 텍스트 블록을 개별 문장으로 분할할 수 있어 긴 텍스트 처리에 특히 유용
* 신속한 텍스트 처리보다 분석적 깊이가 우선시되는 애플리케이션에 가장 적합
* __[KoNLpy Langchain API Doc](https://api.python.langchain.com/en/latest/konlpy/langchain_text_splitters.konlpy.KonlpyTextSplitter.html)__
* Mecab (메캅):
  > 특징: 현재 한국어 NLP에서 가장 성능이 뛰어나고 속도가 빠른 형태소 분석기로 널리 사용. 원래 일본어용으로 개발된 것을 한국어에 맞게 포팅한 '은전한닢' 프로젝트를 통해 활용.<br>
  > 설치: 다른 형태소 분석기에 비해 설치 과정이 다소 번거로울 수 있지만, Colab 등 환경에서는 비교적 쉽게 설치할 수 있음.
* Okt (Open Korean Text):
  > 특징: 비교적 설치가 간편하고 사용하기 쉬움. 초기 프로토타이핑이나 간단한 분석에 적합하며, 사용자 사전 추가도 용이.
* Kkma (꼬꼬마):
  > 특징: 서울대학교에서 개발한 형태소 분석기로, 품사 태그가 세분화되어 있어 학술적인 연구나 정밀한 품사 분석이 필요할 때 유용. 속도는 다소 느린 편.
* Komoran (코모란):
  > 특징: 자바 기반으로 개발되었으며, 비교적 균형 잡힌 성능과 속도를 제공. 

In [ ]:
pip install -qU konlpy

In [ ]:
# data/appendix-keywords.txt 파일을 열어서 f라는 파일 객체를 생성합니다.
with open("./data/appendix-keywords.txt", encoding="UTF-8") as f:
    file = f.read()  # 파일의 내용을 읽어서 file 변수에 저장합니다.

# 파일으로부터 읽은 내용을 일부 출력합니다.
print(file[:1000])

In [ ]:
from langchain_text_splitters import KonlpyTextSplitter

# KonlpyTextSplitter를 사용하여 텍스트 분할기 객체를 생성. (separator: str = "\n\n")
text_splitter = KonlpyTextSplitter(
    chunk_size=200,  # 청크 크기를 200으로 설정합니다.
    chunk_overlap=50,  # 청크 간 중복을 0으로 설정합니다.
)

texts = text_splitter.split_text(file)  # 한국어 문서를 문장 단위로 분할.
print("chunk count ----------------->")
print(len(texts))

#print(texts[0])  # 분할된 문장 중 첫 번째 문장을 출력.

for idx, text in enumerate(texts):
    print("\n" + "[", idx, "] ------------------------------------")
    print(text)